# skillspan-ner: experiment sweep, round 2 (2 GPUs in parallel)

Round 1 (Version 1) results: JobBERT 3 seeds = **0.579 +/- 0.007** F1, **0.598** with Viterbi decoding.
Longer training (lr 3e-5, 8 epochs) and JobSpanBERT gave no gain. DeBERTa crashed (fp16 weights bug, now fixed).

Round 2:

| runs | question |
|---|---|
| `deberta-v3-s42/43/44` | does a stronger general encoder beat JobBERT's domain pretraining? |
| `bert-s43/44` | 2 more seeds for the BERT baseline, so the JobBERT vs BERT gap has error bars |

Note: run outputs in `metrics/` from this version only; combine with round 1 for the full picture.

**Settings:** Accelerator = **GPU T4 x2**, Internet = **On**.

In [ ]:
!git clone -q https://github.com/MSR327/skillspan-ner.git
%cd skillspan-ner
!pip install -q -U "transformers>=5.0" "accelerate>=1.1" datasets sentencepiece
!pip install -q -e ".[train]"

In [ ]:
!nvidia-smi --query-gpu=index,name,memory.total --format=csv

## Pre-download models and data (so the parallel workers don't race on downloads)

In [ ]:
from datasets import load_dataset
from huggingface_hub import snapshot_download

for m in ["jjzha/jobbert-base-cased", "jjzha/jobspanbert-base-cased", "microsoft/deberta-v3-base"]:
    snapshot_download(m, allow_patterns=["*.json", "*.txt", "*.model", "*.safetensors", "*.bin"])
    print("cached", m)
print({k: len(v) for k, v in load_dataset("jjzha/skillspan").items()})

## Run the sweep: one worker per GPU

In [ ]:
import os
import subprocess
import threading
import time
from pathlib import Path

COMMON = ["batch_size=32"]
EXPERIMENTS = {
    # Round 2. Round 1 (Kaggle Version 1): JobBERT seeds 42-44 = 0.579 +/- 0.007 F1
    # (0.598 with Viterbi); lr3e-5/8ep and JobSpanBERT gave no gain; DeBERTa crashed (fp16 bug, fixed).
    0: [  # GPU 0
        ("deberta-v3-s42", ["base_model=microsoft/deberta-v3-base", "learning_rate=3e-5", "seed=42"]),
        ("deberta-v3-s44", ["base_model=microsoft/deberta-v3-base", "learning_rate=3e-5", "seed=44"]),
    ],
    1: [  # GPU 1
        ("deberta-v3-s43", ["base_model=microsoft/deberta-v3-base", "learning_rate=3e-5", "seed=43"]),
        ("bert-s43", ["base_model=bert-base-cased", "seed=43"]),
        ("bert-s44", ["base_model=bert-base-cased", "seed=44"]),
    ],
}
Path("logs").mkdir(exist_ok=True)
status = {}


def worker(gpu, jobs):
    for name, sets in jobs:
        t0 = time.time()
        cmd = ["python", "-m", "skillex.train", "--config", "configs/jobbert.yaml", "--set",
               f"run_name={name}", f"output_dir=outputs/{name}", *COMMON, *sets]
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu)}
        with open(f"logs/{name}.log", "w") as f:
            # check=False: a failed run must not stop the others
            proc = subprocess.run(cmd, check=False, env=env, stdout=f, stderr=subprocess.STDOUT)
        rc = proc.returncode
        status[name] = "OK" if rc == 0 else f"FAILED (rc={rc})"
        print(f"[gpu{gpu}] {name}: {status[name]} in {(time.time() - t0) / 60:.1f} min", flush=True)


threads = [threading.Thread(target=worker, args=(g, jobs)) for g, jobs in EXPERIMENTS.items()]
for t in threads:
    t.start()
for t in threads:
    t.join()
print(status)

## Failures (last 25 log lines of any failed run)

In [ ]:
for name, s in status.items():
    if s != "OK":
        print(f"===== {name} =====")
        print("\n".join(Path(f"logs/{name}.log").read_text().splitlines()[-25:]))

## Results (test set, exact span-match)

In [ ]:
import json

import pandas as pd

rows = []
for f in sorted(Path("metrics").glob("*.json")):
    m = json.loads(f.read_text())
    c = m["config"]
    rows.append({
        "run": f.stem, "model": c["base_model"].split("/")[-1], "seed": c["seed"],
        "lr": f'{c["learning_rate"]:.0e}', "epochs": c["epochs"],
        "f1": m["test_f1"], "vit_f1": m["test_vit_f1"],
        "precision": m["test_precision"], "vit_precision": m["test_vit_precision"],
        "recall": m["test_recall"],
        "f1_skill": m["test_f1_skill"], "vit_f1_skill": m["test_vit_f1_skill"],
        "f1_knowledge": m["test_f1_knowledge"], "vit_f1_knowledge": m["test_vit_f1_knowledge"],
    })
df = pd.DataFrame(rows).set_index("run").sort_values("vit_f1", ascending=False)
pd.set_option("display.width", 250)
print(df.round(4).to_string())

## Mean +/- std per model family (across seeds)

In [ ]:
df["family"] = df.index.str.replace(r"-s\d+$", "", regex=True)
cols = ["f1", "vit_f1", "precision", "vit_precision", "f1_skill", "vit_f1_skill", "f1_knowledge", "vit_f1_knowledge"]
summary = df.groupby("family")[cols].agg(["mean", "std", "count"])
print(summary.round(4).to_string())